In [1]:
import sys
import warnings
from pathlib import Path

from tqdm import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

# Find the FikrLLM repository root, whether Jupyter starts here or in a parent directory.
start = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in (start, *start.parents)
    if (path / "pyproject.toml").exists() and (path / "fikrllm").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT.name)


repo root: FikrLLM


# Tester `fikrllm/tokenizer.py` et les embeddings

Ce notebook est un banc de test interactif pour le fichier `fikrllm/tokenizer.py`. Il vérifie le chargement de `tokenizer.json`, la normalisation, la tokenisation, l'encodage, le décodage et le passage des IDs vers les embeddings, sans construire ni entraîner tout le projet.

In [2]:
from fikrllm.tokenizer import Tokenizer

tok = Tokenizer.from_file()
print("vocab size:", tok.vocab_size)

vocab size: 32000


In [3]:
sentence = "أبوبكر    سليمان"
norm_text = tok.normalize(sentence)
print("Normalized text:", norm_text)

Normalized text: ابوبكر سليمان


In [4]:
encoded = tok.encode(sentence)
print("Encoded:", encoded)

Encoded: [7907, 8051, 8146, 12150]


In [5]:
tokenized = tok.tokenize(sentence)
print("Tokenized:", tokenized)

Tokenized: ['▁اب', 'وب', 'كر', '▁سليمان']


In [6]:
decoded = tok.decode([7907, 8051, 8146, 12150])
print("Decoded:", decoded)

Decoded: ابوبكر سليمان


## Embeddings

In [7]:
from fikrllm.config import ModelConfig
from fikrllm.model.embeddings import InputEmbedding
from fikrllm.tokenizer import Tokenizer
import torch

In [8]:
my_config = ModelConfig()
embed = InputEmbedding(config=my_config)

tok = Tokenizer.from_file()
sentence_1 = "أبوبكر    سليمان Abu Bakr Soliman"
sentence_2 = "أبوبكر    سليمان Abu Bakr Soliman"

In [9]:
encoded_1 = tok.encode(sentence_1)
encoded_2 = tok.encode(sentence_2)

input_ids = torch.tensor([
    encoded_1,
    encoded_2,
])

In [10]:
encoded_1

[7907, 8051, 8146, 12150, 9044, 18246, 14934, 16753]

In [11]:
encoded_2

[7907, 8051, 8146, 12150, 9044, 18246, 14934, 16753]

In [12]:
input_vectors = embed(input_ids)
input_vectors.shape

torch.Size([2, 8, 768])